# Table 3: Pooled GBM Methods Comparison

## Step 1: Run pooled GBM method comparison

This step compares Raw-18, Raw-21, global sigma-star, temperature-dependent sigma-star, and regularized temperature-dependent sigma-star using pooled all-alloy data with 5-fold cross-validation.


In [1]:
"""
Pooled all-data comparison of Raw-18, Raw-21, global sigma*-18, and
temperature-dependent sigma*_T-18 with GBM.
"""

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold

try:
    from numpy.exceptions import RankWarning
except ImportError:
    RankWarning = np.RankWarning

warnings.simplefilter("ignore", RankWarning)


DATA_PATH = os.path.join("..", "Dataset", "Creep_Properties_Dataset.xlsx")

COMP_FEATURES = [
    "C, wt.%",
    "Si, wt.%",
    "Mn, wt.%",
    "P, wt.%",
    "S, wt.%",
    "Ni, wt.%",
    "Cr, wt.%",
    "Mo, wt.%",
    "Al, wt.%",
    "N, wt.%",
    "Ti, wt.%",
    "Fe, wt.%",
    "Nb, wt.%",
    "B, wt.%",
    "Cu, wt.%",
    "Co, wt.%",
]
RAW18_FEATURES = ["Stress (Mpa)", "Temperature of Creep Test (C)"] + COMP_FEATURES
GEO_FEATURES = ["Thickness (mm)", "Cross-Sectional Area", "Aspect Ratio (L/D)"]

GBM_PARAMS = dict(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    min_samples_leaf=3,
    random_state=42,
)
N_SPLITS = 5
LAMBDAS = [0.0, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]

RPV_ALLOYS = {"SA533B", "SA508", "A508/A533"}


def impute_rpv_cross_section(df: pd.DataFrame) -> pd.DataFrame:
    """RPV Cross-Sectional Area."""
    df = df.copy()
    rpv_mask = df["Alloy Designation"].isin(RPV_ALLOYS)
    missing = rpv_mask & df["Cross-Sectional Area"].isna()

    cyl = missing & (df["Specimen Shape"] == "Cylindrical")
    d = df.loc[cyl, "Thickness (mm)"]
    df.loc[cyl, "Cross-Sectional Area"] = np.pi * (d / 2) ** 2
    n_cyl = int(cyl.sum())

    flat = missing & (df["Specimen Shape"] == "Flat")
    rpv_median_overall = df.loc[
        rpv_mask & df["Cross-Sectional Area"].notna(), "Cross-Sectional Area"
    ].median()
    n_flat = 0
    for alloy in RPV_ALLOYS:
        alloy_flat = flat & (df["Alloy Designation"] == alloy)
        if not alloy_flat.any():
            continue
        alloy_median = df.loc[
            rpv_mask
            & (df["Alloy Designation"] == alloy)
            & df["Cross-Sectional Area"].notna(),
            "Cross-Sectional Area",
        ].median()
        fill_val = alloy_median if not np.isnan(alloy_median) else rpv_median_overall
        df.loc[alloy_flat, "Cross-Sectional Area"] = fill_val
        n_flat += int(alloy_flat.sum())

    print(
        f"  [RPV] imputed Cross-Sectional Area: "
        f"{n_cyl} cylindrical (formula), {n_flat} flat (median)"
    )
    return df


def impute_ss316l_composition(df: pd.DataFrame) -> pd.DataFrame:
    """SS316L composition."""
    df = df.copy()
    mask = df["Alloy Designation"] == "SS316L"
    comp_missing = mask & df[COMP_FEATURES[0]].isna()
    if not comp_missing.any():
        return df

    medians = df.loc[mask & ~comp_missing, COMP_FEATURES].median()
    for col in COMP_FEATURES:
        df.loc[comp_missing, col] = medians[col]
    print(
        f"  [SS316L] imputed composition for {int(comp_missing.sum())} records "
        f"using SS316L per-column medians"
    )
    return df


def metrics(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "within_2x_pct": float(
            np.mean(np.abs(y_true - y_pred) < np.log10(2)) * 100.0
        ),
        "within_3x_pct": float(
            np.mean(np.abs(y_true - y_pred) < np.log10(3)) * 100.0
        ),
    }


def proxy(ls_star, y, temperature):
    total = 0.0
    n_total = 0
    for tv in np.unique(temperature):
        mask = temperature == tv
        n = int(np.sum(mask))
        if n < 3:
            continue
        if np.std(ls_star[mask]) < 1e-10:
            resid = y[mask] - np.mean(y[mask])
        else:
            try:
                coef = np.polyfit(ls_star[mask], y[mask], 1)
                resid = y[mask] - np.polyval(coef, ls_star[mask])
            except np.linalg.LinAlgError:
                resid = y[mask] - np.mean(y[mask])
        total += float(np.var(resid)) * n
        n_total += n
    return total / max(n_total, 1)


def apply_global(params, ls, lb, l_area, l_ld):
    out = ls + params[0] * lb + params[1] * l_area + params[2] * l_ld
    return np.clip(out, -1.0, 4.0)


def apply_tdep(params, ls, t_norm, lb, l_area, l_ld):
    out = (
        ls
        + (params[0] + params[3] * t_norm) * lb
        + (params[1] + params[4] * t_norm) * l_area
        + (params[2] + params[5] * t_norm) * l_ld
    )
    return np.clip(out, -1.0, 4.0)


def opt_global(ls, y, temperature, lb, l_area, l_ld):
    res = minimize(
        lambda p: proxy(apply_global(p, ls, lb, l_area, l_ld), y, temperature),
        x0=[0.0, 0.0, -0.1],
        method="Nelder-Mead",
        options={"maxiter": 3000, "xatol": 1e-5, "fatol": 1e-6},
    )
    return res.x


def opt_tdep_regularized(ls, y, temperature, t_norm, lb, l_area, l_ld, p0, lam):
    x0 = [p0[0], p0[1], p0[2], 0.0, 0.0, 0.0]

    def objective(params):
        ls_star = apply_tdep(params, ls, t_norm, lb, l_area, l_ld)
        return proxy(ls_star, y, temperature) + lam * float(np.mean((ls_star - ls) ** 2))

    res = minimize(
        objective,
        x0=x0,
        method="Nelder-Mead",
        options={"maxiter": 5000, "xatol": 1e-5, "fatol": 1e-6},
    )
    return res.x


def replace_stress(x, ls_star):
    out = x.copy()
    out[:, 0] = 10.0 ** ls_star
    return out


def fit_predict(x, y, train_idx, val_idx):
    model = GradientBoostingRegressor(**GBM_PARAMS)
    model.fit(x[train_idx], y[train_idx])
    return model.predict(x[val_idx])


def load_data():
    print("Loading data ...")
    df = pd.read_excel(DATA_PATH, sheet_name="Data", header=1)
    df = impute_rpv_cross_section(df)
    df = impute_ss316l_composition(df)
    mask = (
        df["Alloy Designation"].notna()
        & (df["Notch"] == "unnotched")
        & df["Creep Rupture Life (h)"].notna()
        & (df["Creep Rupture Life (h)"] > 0)
        & df["Stress (Mpa)"].notna()
        & (df["Stress (Mpa)"] > 0)
        & df["Specimen Shape"].isin(["Flat", "Cylindrical"])
    )
    data = df[mask].copy().reset_index(drop=True)
    data["log_tR"] = np.log10(data["Creep Rupture Life (h)"])
    data["log_sigma"] = np.log10(data["Stress (Mpa)"])

    required = RAW18_FEATURES + ["Alloy Designation", "log_tR", "log_sigma"]
    sub = data[required + GEO_FEATURES].dropna(subset=required).reset_index(drop=True)
    dummies = pd.get_dummies(sub["Alloy Designation"], prefix="alloy").astype(float)

    x_raw18 = np.column_stack([sub[RAW18_FEATURES].values.astype(float), dummies.values])
    y = sub["log_tR"].values.astype(float)
    ls = sub["log_sigma"].values.astype(float)
    temperature = sub["Temperature of Creep Test (C)"].values.astype(float)
    t_norm = (temperature - float(np.mean(temperature))) / float(np.std(temperature))

    b = pd.to_numeric(sub["Thickness (mm)"], errors="coerce").values.astype(float)
    area = pd.to_numeric(sub["Cross-Sectional Area"], errors="coerce").values.astype(float)
    ld = pd.to_numeric(sub["Aspect Ratio (L/D)"], errors="coerce").values.astype(float)

    valid_b = np.isfinite(b) & (b > 0)
    valid_area = np.isfinite(area) & (area > 0)
    valid_ld = np.isfinite(ld) & (ld > 0)
    b0 = float(np.median(b[valid_b]))
    area0 = float(np.median(area[valid_area]))
    ld0 = float(np.median(ld[valid_ld]))

    b_imputed = b.copy()
    area_imputed = area.copy()
    ld_imputed = ld.copy()
    b_imputed[~valid_b] = b0
    area_imputed[~valid_area] = area0
    ld_imputed[~valid_ld] = ld0
    x_raw21 = np.column_stack(
        [
            sub[RAW18_FEATURES].values.astype(float),
            b_imputed,
            area_imputed,
            ld_imputed,
            dummies.values,
        ]
    )

    lb = np.zeros(len(sub))
    l_area = np.zeros(len(sub))
    l_ld = np.zeros(len(sub))
    lb[valid_b] = np.log10(b[valid_b] / b0)
    l_area[valid_area] = np.log10(area[valid_area] / area0)
    l_ld[valid_ld] = np.log10(ld[valid_ld] / ld0)

    missing_any_geo = ~(valid_b & valid_area & valid_ld)
    print(f"  Samples: {len(y)}")
    print(f"  Alloys: {dummies.shape[1]}")
    print(f"  Geometry-missing rows with neutral correction: {int(np.sum(missing_any_geo))}")
    print(f"  References: b0={b0:.4g}, A0={area0:.4g}, LD0={ld0:.4g}")

    return (
        x_raw18,
        x_raw21,
        y,
        ls,
        temperature,
        t_norm,
        lb,
        l_area,
        l_ld,
        dummies.shape[1],
        int(np.sum(missing_any_geo)),
    )


def run_table3_pooled_gbm_methods():
    (
        x_raw18,
        x_raw21,
        y,
        ls,
        temperature,
        t_norm,
        lb,
        l_area,
        l_ld,
        n_alloys,
        n_missing_geo,
    ) = load_data()
    n = len(y)
    splits = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=42).split(x_raw18))

    preds = {
        "Raw-18 + alloy": np.zeros(n),
        "Raw-21 + alloy": np.zeros(n),
        "Global sigma*-18 + alloy": np.zeros(n),
    }
    for lam in LAMBDAS:
        label = (
            "T-dependent sigma*_T-18 + alloy"
            if lam == 0.0
            else f"Reg T-dependent sigma*_T-18 + alloy lambda={lam:g}"
        )
        preds[label] = np.zeros(n)

    print("\nRunning 5-fold CV ...")
    for fold, (train_idx, val_idx) in enumerate(splits, 1):
        print(f"  Fold {fold}/{N_SPLITS}")
        preds["Raw-18 + alloy"][val_idx] = fit_predict(x_raw18, y, train_idx, val_idx)
        preds["Raw-21 + alloy"][val_idx] = fit_predict(x_raw21, y, train_idx, val_idx)

        p_global = opt_global(
            ls[train_idx],
            y[train_idx],
            temperature[train_idx],
            lb[train_idx],
            l_area[train_idx],
            l_ld[train_idx],
        )
        ls_global = apply_global(p_global, ls, lb, l_area, l_ld)
        preds["Global sigma*-18 + alloy"][val_idx] = fit_predict(
            replace_stress(x_raw18, ls_global), y, train_idx, val_idx
        )
        for lam in LAMBDAS:
            p_tdep = opt_tdep_regularized(
                ls[train_idx],
                y[train_idx],
                temperature[train_idx],
                t_norm[train_idx],
                lb[train_idx],
                l_area[train_idx],
                l_ld[train_idx],
                p_global,
                lam,
            )
            ls_star = apply_tdep(p_tdep, ls, t_norm, lb, l_area, l_ld)
            x_star = replace_stress(x_raw18, ls_star)
            label = (
                "T-dependent sigma*_T-18 + alloy"
                if lam == 0.0
                else f"Reg T-dependent sigma*_T-18 + alloy lambda={lam:g}"
            )
            preds[label][val_idx] = fit_predict(x_star, y, train_idx, val_idx)

    raw_r2 = metrics(y, preds["Raw-18 + alloy"])["r2"]
    rows = []
    for label, pred in preds.items():
        m = metrics(y, pred)
        lam = np.nan
        if "lambda=" in label:
            lam = float(label.split("lambda=")[-1])
        elif label.startswith("T-dependent"):
            lam = 0.0
        rows.append(
            {
                "method": label,
                "lambda": lam,
                "n": n,
                "n_alloys": n_alloys,
                "n_neutral_geometry_rows": n_missing_geo,
                "r2": m["r2"],
                "rmse": m["rmse"],
                "within_2x_pct": m["within_2x_pct"],
                "within_3x_pct": m["within_3x_pct"],
                "delta_vs_raw18": m["r2"] - raw_r2,
            }
        )

    long_df = pd.DataFrame(rows)
    best_reg = (
        long_df[long_df["method"].str.startswith("Reg")]
        .sort_values("r2", ascending=False)
        .head(1)
    )
    summary = pd.concat(
        [
            long_df[long_df["method"] == "Raw-18 + alloy"],
            long_df[long_df["method"] == "Raw-21 + alloy"],
            long_df[long_df["method"] == "Global sigma*-18 + alloy"],
            long_df[long_df["method"] == "T-dependent sigma*_T-18 + alloy"],
            best_reg,
        ],
        ignore_index=True,
    )

    print("\nSummary:")
    print(summary.to_string(index=False))

    return long_df, summary


In [2]:
# Run Step 1
table3_long, table3_summary = run_table3_pooled_gbm_methods()


Loading data ...
  [RPV] imputed Cross-Sectional Area: 15 cylindrical (formula), 47 flat (median)
  [SS316L] imputed composition for 15 records using SS316L per-column medians
  Samples: 2433
  Alloys: 12
  Geometry-missing rows with neutral correction: 0
  References: b0=8, A0=48, LD0=5

Running 5-fold CV ...
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5

Summary:
                                          method  lambda    n  n_alloys  n_neutral_geometry_rows       r2     rmse  within_2x_pct  within_3x_pct  delta_vs_raw18
                                  Raw-18 + alloy     NaN 2433        12                        0 0.904042 0.358233      74.106042      88.614879        0.000000
                                  Raw-21 + alloy     NaN 2433        12                        0 0.906639 0.353353      74.064940      88.655980        0.002597
                        Global sigma*-18 + alloy     NaN 2433        12                        0 0.908868 0.349109      74.640362      89.10

## Final Table 3

This step formats the Step 1 summary into the paper-ready Table 3.


In [3]:
METHOD_LABELS = {
    "Raw-18 + alloy": "Raw-18",
    "Raw-21 + alloy": "Raw-21",
    "Global sigma*-18 + alloy": "Global sigma*-18",
    "T-dependent sigma*_T-18 + alloy": "sigma*_T-18",
}


def method_label(method: str) -> str:
    if method.startswith("Reg T-dependent sigma*_T-18"):
        return "Reg sigma*_T-18"
    return METHOD_LABELS.get(method, method)


def format_delta(value: float) -> str:
    if abs(value) < 0.00005:
        return "0"
    return f"{value:+.4f}"


final_table = table3_summary.copy()
final_table["Method"] = final_table["method"].map(method_label)
final_table["R2"] = final_table["r2"].map(lambda value: f"{value:.4f}")
final_table["RMSE"] = final_table["rmse"].map(lambda value: f"{value:.4f}")
final_table["Delta vs Raw-18"] = final_table["delta_vs_raw18"].map(format_delta)
final_table = final_table[["Method", "R2", "RMSE", "Delta vs Raw-18"]].reset_index(drop=True)
final_table

,Method,R2,RMSE,Delta vs Raw-18
0,Raw-18,0.9040,0.3582,0
1,Raw-21,0.9066,0.3534,+0.0026
2,Global sigma*-18,0.9089,0.3491,+0.0048
3,sigma*_T-18,0.9138,0.3396,+0.0097
4,Reg sigma*_T-18,0.9142,0.3387,+0.0102
